# date_parser 실제 OCR 연동 데모 (개인 검증용)

담당: 이수민

`date_parser_dev.ipynb`는 OCR이 없는, 텍스트만 가짜로 만든 검증용이었습니다.
이 노트북은 **실제 OCR 엔진을 돌려서** 나온 결과를 공통 형식
(`{text, confidence, bbox}`)으로 변환한 뒤, **진짜로 `date_parser`를 import해서**
소비기한을 뽑아내는 end-to-end 데모입니다.

**PaddleOCR 사용**: 오늘 22장 표본 비교 실험 결과 PaddleOCR이 EasyOCR보다
찾음 개수가 더 많았어서(PaddleOCR 15/22 vs EasyOCR 9~11/22) 여기서도
PaddleOCR을 기준으로 씁니다. 아래 초기화 설정은 오늘 직접 디버깅해서
맞춘 그 설정 그대로입니다 (`enable_mkldnn=False`, 모델 이름/경로 명시 등).
PaddleOCR이 이미지당 더 느린 건 사실이라(오늘 기준 장당 약 34초 vs EasyOCR
약 10~23초), 최종 파이프라인에서 어떤 엔진을 쓸지는 정서현 담당이며 팀
논의로 정할 사안입니다 — 여기서는 어디까지나 "date_parser가 실제 OCR
출력과 잘 맞물리는지" 확인용입니다.

⚠️ OCR 엔진 자체는 이수민 담당이 아닙니다 (정서현 담당). 여기서는 본인
모듈이 실제 OCR 출력과 맞물려 잘 동작하는지 개인적으로 확인하기 위한
용도로만 PaddleOCR을 임시로 붙였습니다.

⚠️ 이 노트북은 `notebooks/` 폴더에 있으므로 대회 채점 대상이 아닙니다
(채점 대상은 `predict.ipynb` 하나뿐, 팀 공식 규정 5번 참고).

## 실행 전 준비물 (본인 PC 기준)
- 오늘 이미 구축한 `itda` conda 환경 + PaddleOCR (`weights/paddleocr/` 폴더 포함)
- 실제 상품 이미지가 들어있는 폴더
- 아래 `INPUT_DIR`, `WEIGHTS_DIR`를 본인 PC의 실제 경로로 수정

## 1. 설정

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "date_parser").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("date_parser 패키지를 찾을 수 없습니다. 프로젝트 루트에서 실행하세요.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ===== 본인 환경에 맞게 수정 =====
INPUT_DIR = Path("./val_images")        # 실제 이미지 폴더 경로로 변경
WEIGHTS_DIR = Path("./weights/paddleocr")  # 오늘 다운로드한 PaddleOCR 가중치 폴더
MAX_IMAGES = 10                          # 테스트로 몇 장만 돌려볼지 (전체 실행 시 None)
# =================================

print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. PaddleOCR 초기화

오늘 디버깅해서 맞춘 설정 그대로입니다.

In [ ]:
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

from paddleocr import PaddleOCR

ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_detection_model_dir=str(WEIGHTS_DIR / "PP-OCRv5_mobile_det_infer"),
    text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
    text_recognition_model_dir=str(WEIGHTS_DIR / "korean_PP-OCRv5_mobile_rec_infer"),
    use_doc_orientation_classify=True,
    doc_orientation_classify_model_name="PP-LCNet_x1_0_doc_ori",
    doc_orientation_classify_model_dir=str(WEIGHTS_DIR / "PP-LCNet_x1_0_doc_ori_infer"),
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=False,
)
print("PaddleOCR 로딩 완료")

## 3. 공통 형식으로 변환

`ocr.predict()`가 반환하는 결과는 `rec_polys`/`rec_texts`/`rec_scores` 키를 가진
dict 형태입니다. 이걸 팀에서 정한 공통 인터페이스 `{"text":..., "confidence":..., "bbox":...}`
딕셔너리 리스트로 바꾸는 부분만 여기서 담당하고, 그 다음부터는 순수하게
`date_parser` 모듈이 처리합니다.

In [ ]:
def paddleocr_to_common_format(res):
    """PaddleOCR predict() 결과 1건 -> [{"text","confidence","bbox"}, ...]"""
    return [
        {"text": text, "confidence": float(conf), "bbox": [[float(x), float(y)] for x, y in poly]}
        for poly, text, conf in zip(res["rec_polys"], res["rec_texts"], res["rec_scores"])
    ]


# 변환 함수만 먼저 가짜 데이터로 검증 (실제 OCR 없이도 바로 확인 가능)
_fake_paddle_result = {
    "rec_polys": [
        [[0, 100], [60, 100], [60, 115], [0, 115]],
        [[0, 118], [90, 118], [90, 133], [0, 133]],
    ],
    "rec_texts": ["소비기한", "2026.07.15"],
    "rec_scores": [0.95, 0.90],
}
paddleocr_to_common_format(_fake_paddle_result)

## 4. date_parser 실제 import & end-to-end 실행

여기서부터가 실제로 본인 모듈이 쓰이는 부분입니다.

In [ ]:
from date_parser import parse_expiration_date

import glob
import time

import pandas as pd

IMAGE_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
image_paths = sorted({p for ext in IMAGE_EXTS for p in glob.glob(str(INPUT_DIR / ext))})
if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

print(f"대상 이미지 {len(image_paths)}장")

rows = []
start = time.time()
for path in image_paths:
    ocr_raw = ocr.predict(path, text_det_limit_side_len=1280, text_det_limit_type="max")
    ocr_common = paddleocr_to_common_format(ocr_raw[0])
    result = parse_expiration_date(ocr_common)
    rows.append({"image_id": Path(path).stem, **result})

elapsed = time.time() - start
result_df = pd.DataFrame(rows, columns=["image_id", "year", "month", "day", "final_date"])
found = (result_df["final_date"] != "NONE").sum()
print(f"총 {len(image_paths)}장 | 찾음 {found} | 걸린 시간 {elapsed:.1f}초")
result_df

## 5. 결과 저장 (선택)

라벨링된 데이터가 있다면 `label(1).csv`/`label2.xlsx`와 비교해서 정확도를
직접 확인해볼 수 있습니다. (이 노트북에서는 채점용 `submission.csv`를
만들지 않습니다 — 그건 최종 통합된 `predict.ipynb`가 하는 일입니다.)

In [ ]:
result_df.to_csv("date_parser_check.csv", index=False, encoding="utf-8-sig")
print("저장 완료: date_parser_check.csv")